# Fra 2d-liste til dictionary (revidert)

Et vanlig tilfelle: vi har data i ei 2d-liste, og vil gjøre om til en dictionary for å slå opp raskt på en **nøkkel**.

**Arbeidsmåte:** Skriv ned en prediksjon på spørsmålene før du kjører cellene.

## 1. Motivasjon: hvorfor ikke bare bruke 2d-lista?

Vi har info om fire grunnstoff: atomnummer, navn, kjemisk symbol og atommasse.

In [ ]:
grunnstoff = [
    ['1', 'Hydrogen', 'H', '1.0079'],
    ['2', 'Helium', 'He', '4.0026'],
    ['3', 'Litium', 'Li', '6.941'],
    ['4', 'Beryllium', 'Be', '9.0122'],
]

**Oppslag på atomnummer** er enkelt, fordi atomnummer 1 ligger på indeks 0 osv.:

In [ ]:
atomnr = 2
print(grunnstoff[atomnr - 1])

**Spørsmål:** Hvordan finner vi raden for symbolet `'He'`? Indeksen forteller oss ingenting her, så vi må lete gjennom lista rad for rad:

In [ ]:
def finn_symbol(symbol):
    for rad in grunnstoff:
        if rad[2] == symbol:
            return rad
    return None

print(finn_symbol('He'))

Dette er **O(n)**: i verste fall må vi se på alle radene, for *hvert* oppslag. Med en dictionary med symbolet som nøkkel blir oppslaget i gjennomsnitt **O(1)**:

```python
grunnstoff_dict['He']
```

Poenget med å gjøre om 2d-lista til en dictionary: velg det du vil slå opp på som nøkkel.

## 2. Enkelt tilfelle: nøkkelen er unik

Vi skal lage funksjonen `lag_grunnstoff_dict(grunnstoff)` som får inn lista og returnerer en dictionary med symbolet som nøkkel.

Når vi vet at det er nøyaktig én rad per grunnstoff, kan vi gå gjennom lista og lage ett element per runde.

Vi pakker først ut raden til navngitte variabler. Det er lettere å lese enn `rad[0:2] + rad[3:]`.

In [ ]:
def lag_grunnstoff_dict(grunnstoff):
    """Returnerer {symbol: [atomnummer, navn, atommasse]}."""
    grunnstoff_dict = {}
    for atomnr, navn, symbol, masse in grunnstoff:
        grunnstoff_dict[symbol] = [atomnr, navn, masse]
    return grunnstoff_dict

grunnstoff_dict = lag_grunnstoff_dict(grunnstoff)
print(grunnstoff_dict)
print(grunnstoff_dict['He'])

**Merk:** Tallene er lagret som strenger (`'1'`, `'1.0079'`) i denne lista. Vil vi regne med dem, må vi konvertere til `int` og `float`:

```python
grunnstoff_dict[symbol] = [int(atomnr), navn, float(masse)]
```

## 3. Komplikasjon: samme nøkkel forekommer flere ganger

Bompasseringer: vi vil ha skiltnummer som nøkkel. Noen biler har flere passeringer, og vi vil ta vare på alle.

In [ ]:
bompasseringer = [
    ['020323', '10:35:02', 'GX34349'],
    ['020323', '10:35:55', 'GD30009'],
    ['020323', '10:35:02', 'GX34349'],
    ['020323', '10:41:02', 'GY19911'],
    ['030323', '09:13:05', 'JX10000'],
    ['030323', '09:23:00', 'GD30009'],
]

Vi prøver samme fremgangsmåte som i forrige eksempel.

**Spørsmål før du kjører:** Hva tror du dictionary-en inneholder for `'GX34349'` og `'GD30009'`? Hvor mange passeringer har hver av dem i lista?

In [ ]:
def lag_bom_dict_v1(bompasseringer):
    passeringer_per_bil = {}
    for dato, tid, skilt in bompasseringer:
        passeringer_per_bil[skilt] = [(dato, tid)]
    return passeringer_per_bil

print(lag_bom_dict_v1(bompasseringer))

Problemet er at andre forekomst av en bil **overskriver** det vi hadde fra før. Vi ender med bare én passering per bil.

Vi må derfor teste for hver rad:

| Nøkkelen finnes... | Hva gjør vi? |
|---|---|
| **ikke** fra før | Lag en ny nøkkel med ei liste som inneholder denne passeringen |
| allerede | Legg passeringen bakerst i lista som ligger der |

**Spørsmål:** Hva blir dictionary-en etter rad 1, etter rad 2 og etter rad 3? Tegn den.

In [ ]:
def lag_bom_dict_v2(bompasseringer):
    """Returnerer {skiltnummer: [(dato, tid), ...]}."""
    passeringer_per_bil = {}
    for dato, tid, skilt in bompasseringer:
        passering = (dato, tid)
        if skilt in passeringer_per_bil:
            passeringer_per_bil[skilt].append(passering)
        else:
            passeringer_per_bil[skilt] = [passering]
    return passeringer_per_bil

print(lag_bom_dict_v2(bompasseringer))

### Variant med `not in`

Vi kan snu testen. Da kommer opprettelsen først, og vi kan droppe `else`. Siden vi uansett legger til passeringen etterpå, må vi opprette en **tom** liste:

In [ ]:
def lag_bom_dict_v3(bompasseringer):
    passeringer_per_bil = {}
    for dato, tid, skilt in bompasseringer:
        if skilt not in passeringer_per_bil:
            passeringer_per_bil[skilt] = []
        passeringer_per_bil[skilt].append((dato, tid))
    return passeringer_per_bil

print(lag_bom_dict_v3(bompasseringer))

Begge variantene gjør det samme. `v3` er litt kortere og unngår å gjenta opprettelsen av passeringen.

### Snarveier

`dict.setdefault(nøkkel, standard)` returnerer verdien hvis nøkkelen finnes, og oppretter den med standardverdien ellers. Da blir løkka bare én linje:

In [ ]:
def lag_bom_dict_v4(bompasseringer):
    passeringer_per_bil = {}
    for dato, tid, skilt in bompasseringer:
        passeringer_per_bil.setdefault(skilt, []).append((dato, tid))
    return passeringer_per_bil

print(lag_bom_dict_v4(bompasseringer))

`collections.defaultdict` gjør det samme, men oppretter standardverdien automatisk. Dette er ekstra, og ikke eksamensstoff:

In [ ]:
from collections import defaultdict

def lag_bom_dict_v5(bompasseringer):
    passeringer_per_bil = defaultdict(list)
    for dato, tid, skilt in bompasseringer:
        passeringer_per_bil[skilt].append((dato, tid))
    return dict(passeringer_per_bil)

print(lag_bom_dict_v5(bompasseringer))

## Oppsummering

Fra 2d-liste til dictionary, typisk fremgangsmåte:

1. Bestem hvilket element i raden som skal være **nøkkel**.
2. Gå gjennom lista rad for rad og pakk ut raden til navngitte variabler.
3. For hver rad, sjekk om nøkkelen finnes fra før:
   - **ja:** legg sammen med det vi allerede har på nøkkelen (f.eks. `append`)
   - **nei:** opprett et nytt element med tilhørende data
4. Hvis hver nøkkel bare forekommer én gang, trenger vi ikke testen.

Komplikasjoner kan forekomme, f.eks. at ikke alle rader skal tas med. Det ser vi ikke på her.

## Utfordringer

Bruk `passeringer_per_bil = lag_bom_dict_v3(bompasseringer)`:

1. Skriv ut antall passeringer per bil.
2. Finn skiltnummeret til bilen med flest passeringer. (Hint: `max(d, key=...)`)
3. Passeringen for `GX34349` kl. 10:35:02 står to ganger. Er det én passering registrert to ganger, eller to passeringer? Hvordan kan vi fjerne duplikater? (Hint: `set`)
4. Lag en dictionary der nøkkelen er **dato** i stedet, og verdien er ei liste med skiltnumre den dagen. Hva endrer seg i koden?
5. Slå opp `'He'` i `grunnstoff_dict` og skriv ut navnet. Hva skjer ved oppslag på `'Xx'`? Hvordan kan `get` hjelpe?